# RTL Simulation: Distraction Alert Controller + Camera Link

Runs the two self-checking testbenches for real, using Vivado 2017.4's command-line simulator (`xvlog` -> `xelab` -> `xsim`), and shows the actual pass/fail output. Nothing here is copy-pasted from a log file - each cell below invokes the real tool and captures its real output.

1. `distraction_alert_controller_tb.v` - unit test for the alert/debounce logic on its own, with synthetic stimulus.
2. `camera_link_top_tb.v` - end-to-end test: a real UART byte stream reproducing 28 real predictions from the actual trained model (`experiments/06_cross_view_mobilenetv2/best_mobilenetv2_crossview.keras`) on real test images, bit-banged through `uart_rx` -> `camera_link_top` -> `distraction_alert_controller`.

See `fpga/README.md` for the design rationale and what this component is/isn't.

In [1]:
import subprocess
from pathlib import Path

FPGA_DIR = Path("..").resolve()
VIVADO_SETTINGS = r"D:\Apps\Vivado\Vivado_Program\Vivado\2017.4\settings64.bat"


def run_vivado_cmd(cmd, cwd):
    """Run a Vivado command-line tool with the environment set up, capture output."""
    full_cmd = f'call "{VIVADO_SETTINGS}" && {cmd}'
    result = subprocess.run(
        full_cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=300
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:\n", result.stderr)
    return result

## 1. Alert controller unit test

In [2]:
sim_dir = FPGA_DIR / "sim"

run_vivado_cmd(
    "xvlog ..\\rtl\\distraction_alert_controller.v distraction_alert_controller_tb.v",
    cwd=sim_dir,
)
run_vivado_cmd(
    "xelab -debug typical distraction_alert_controller_tb -s dac_sim",
    cwd=sim_dir,
)
result = run_vivado_cmd("xsim dac_sim -runall", cwd=sim_dir)

assert "ALL CHECKS PASSED" in result.stdout, "Testbench did not report all checks passed - see output above"
print("\nUnit test: ALL CHECKS PASSED, confirmed from this run's own output.")

INFO: [VRFC 10-2263] Analyzing Verilog file "D:/MSC_PROJECT/fpga/rtl/distraction_alert_controller.v" into library work
INFO: [VRFC 10-311] analyzing module distraction_alert_controller
INFO: [VRFC 10-2263] Analyzing Verilog file "D:/MSC_PROJECT/fpga/sim/distraction_alert_controller_tb.v" into library work
INFO: [VRFC 10-311] analyzing module distraction_alert_controller_tb



Vivado Simulator 2017.4
Copyright 1986-1999, 2001-2016 Xilinx, Inc. All Rights Reserved.
Running: D:/Apps/Vivado/Vivado_Program/Vivado/2017.4/bin/unwrapped/win64.o/xelab.exe -debug typical distraction_alert_controller_tb -s dac_sim 
Multi-threading is on. Using 10 slave threads.
Starting static elaboration
Completed static elaboration
Starting simulation data flow analysis
Completed simulation data flow analysis
Time Resolution for simulation is 1ps
Compiling module work.distraction_alert_controller
Compiling module work.distraction_alert_controller_tb
Built simulation snapshot dac_sim




****** xsim v2017.4 (64-bit)
  **** SW Build 2086221 on Fri Dec 15 20:55:39 MST 2017
  **** IP Build 2085800 on Fri Dec 15 22:25:07 MST 2017
    ** Copyright 1986-2017 Xilinx, Inc. All Rights Reserved.

source xsim.dir/dac_sim/xsim_script.tcl
# xsim {dac_sim} -autoloadwcfg -runall
Vivado Simulator 2017.4
Time resolution is 1 ps
run -all
PASS [15000] after reset : alert=0 as expected
PASS [116000] after 10 confident safe ticks : alert=0 as expected
PASS [146000]  (transient, should be rejected) : alert=0 as expected
PASS [156000]  to safe following a short burst : alert=0 as expected
PASS [226000] ck before ALERT_ON_COUNT reached : alert=0 as expected
PASS [236000]  should not itself trigger alert : alert=0 as expected
PASS [246000] rted after sustained distraction : alert=1 as expected
PASS [296000] ted during continued distraction : alert=1 as expected
PASS [366000] reached, alert should still hold : alert=1 as expected
PASS [376000] red after sustained safe driving : alert=0 as expe

## 2. End-to-end test: real model predictions over a simulated UART link

Stimulus is `fpga/reports/real_model_session_sequence.csv` - 28 real `(class_id, confidence)` pairs the trained model actually produced on real test images, not synthetic data.

In [3]:
run_vivado_cmd(
    "xvlog ..\\rtl\\uart_rx.v ..\\rtl\\distraction_alert_controller.v ..\\rtl\\camera_link_top.v camera_link_top_tb.v",
    cwd=sim_dir,
)
run_vivado_cmd(
    "xelab -debug typical camera_link_top_tb -s camera_link_sim",
    cwd=sim_dir,
)
result = run_vivado_cmd("xsim camera_link_sim -runall", cwd=sim_dir)

assert "ALL CHECKS PASSED" in result.stdout, "Testbench did not report all checks passed - see output above"
print("\nEnd-to-end test: ALL CHECKS PASSED, confirmed from this run's own output.")

INFO: [VRFC 10-2263] Analyzing Verilog file "D:/MSC_PROJECT/fpga/rtl/uart_rx.v" into library work
INFO: [VRFC 10-311] analyzing module uart_rx
INFO: [VRFC 10-2263] Analyzing Verilog file "D:/MSC_PROJECT/fpga/rtl/distraction_alert_controller.v" into library work
INFO: [VRFC 10-311] analyzing module distraction_alert_controller
INFO: [VRFC 10-2263] Analyzing Verilog file "D:/MSC_PROJECT/fpga/rtl/camera_link_top.v" into library work
INFO: [VRFC 10-311] analyzing module camera_link_top
INFO: [VRFC 10-2263] Analyzing Verilog file "D:/MSC_PROJECT/fpga/sim/camera_link_top_tb.v" into library work
INFO: [VRFC 10-311] analyzing module camera_link_top_tb



Vivado Simulator 2017.4
Copyright 1986-1999, 2001-2016 Xilinx, Inc. All Rights Reserved.
Running: D:/Apps/Vivado/Vivado_Program/Vivado/2017.4/bin/unwrapped/win64.o/xelab.exe -debug typical camera_link_top_tb -s camera_link_sim 
Multi-threading is on. Using 10 slave threads.
Starting static elaboration
Completed static elaboration
Starting simulation data flow analysis
Completed simulation data flow analysis
Time Resolution for simulation is 1ps
Compiling module work.uart_rx_default
Compiling module work.distraction_alert_controller
Compiling module work.camera_link_top
Compiling module work.camera_link_top_tb
Built simulation snapshot camera_link_sim




****** xsim v2017.4 (64-bit)
  **** SW Build 2086221 on Fri Dec 15 20:55:39 MST 2017
  **** IP Build 2085800 on Fri Dec 15 22:25:07 MST 2017
    ** Copyright 1986-2017 Xilinx, Inc. All Rights Reserved.

source xsim.dir/camera_link_sim/xsim_script.tcl
# xsim {camera_link_sim} -autoloadwcfg -runall
Vivado Simulator 2017.4
Time resolution is 1 ps
run -all
PASS [1050325000] mples 0-5 (initial safe driving) : alert=0 as expected
PASS [3133525000] les 6-17 (sustained distraction) : alert=1 as expected
PASS [4348725000] o safe, alert should still hold) : alert=1 as expected
PASS [4522325000] mple 25 (alert should now clear) : alert=0 as expected
PASS [4695925000] ion blip, should not re-trigger) : alert=0 as expected
PASS [4869525000] ter real sample 27 (final state) : alert=0 as expected

TEST RESULT: ALL CHECKS PASSED (0 errors) - full real-data session correctly handled end to end
$finish called at time : 4869525 ns : File "D:/MSC_PROJECT/fpga/sim/camera_link_top_tb.v" Line 162
exit
INFO:

## What this proves

The alert controller correctly debounces noisy classifications (a 3-tick transient burst doesn't trigger it) and correctly absorbs two genuine misclassifications the real model made on real images, without ever falsely triggering. This is the decision/alert stage only - not the CNN itself running on FPGA. See `fpga/README.md` for the full scope statement.